# Model Training & Evaluation Notebook

## 1. Objective
Train multiple regression models to predict `math_score`, evaluate their metrics (MAE, RMSE, R²), select the best model, and save the fitted preprocessor and trained model as artifacts for deployment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

# Modeling & Evaluation Imports
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings('ignore')

## 2. Load Dataset & Define X, y

In [ ]:
df = pd.read_csv('../data/stud.csv')
X = df.drop(columns=['math_score'], axis=1)
y = df['math_score']

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)

## 3. Preprocessing Setup (ColumnTransformer)
- Categorical Features -> OneHotEncoder
- Numerical Features -> StandardScaler

In [ ]:
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features),
    ]
)

X = preprocessor.fit_transform(X)
print("Transformed X shape:", X.shape)

## 4. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## 5. Evaluation Function

In [ ]:
def eval_metrics(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

## 6. Train & Benchmark Multiple Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(),
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}

model_list = []
r2_list = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    train_mae, train_rmse, train_r2 = eval_metrics(y_train, y_train_pred)
    test_mae, test_rmse, test_r2 = eval_metrics(y_test, y_test_pred)
    
    print(f"=== {name} ===")
    print("Model performance for Training set:")
    print(f"- RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f} | R2 Score: {train_r2:.4f}")
    print("Model performance for Testing set:")
    print(f"- RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f} | R2 Score: {test_r2:.4f}\n")
    
    model_list.append(name)
    r2_list.append(test_r2)

## 7. Model Ranking

In [ ]:
results_df = pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"], ascending=False)
results_df

## 8. Select & Inspect Best Model (Linear Regression / Ridge)

In [ ]:
best_model = LinearRegression()
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
score = r2_score(y_test, y_pred)
print(f"Accuracy of the model is {score * 100:.2f}%")

pred_df = pd.DataFrame({'Actual Value': y_test, 'Predicted Value': y_pred, 'Difference': y_test - y_pred})
pred_df.head(10)

In [ ]:
# Save artifacts
os.makedirs('../artifacts', exist_ok=True)
pickle.dump(preprocessor, open('../artifacts/preprocessor.pkl', 'wb'))
pickle.dump(best_model, open('../artifacts/model.pkl', 'wb'))
print("Artifacts preprocessor.pkl and model.pkl saved successfully!")